[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gaoithee/PML26/blob/main/03_causality_in_nns.ipynb)

> ⚠️ **GPU required.**
> You can either run this notebook in Colab (enable a GPU via **Runtime → Change runtime type → GPU (T4)**), or run it locally if your machine has a compatible GPU.
>
> If running locally, install the required libraries with:
> `pip install -r requirements.txt`


---

# Opening the Black-Boxes: Mechanistic Interpretability of Neural Networks

Modern neural networks are often presented as wonderful black-boxes of amazing capabilities.
But if we peel back the layers, we find a much older and elegant truth.

Neural Networks can be seen as **computational graphs** — systems of nodes (units) and edges
(interactions), where global behaviour emerges from local relations.

This notebook guides you through the historical lineage of these ideas, from 1940s
neuroscience to modern Transformer mechanistic interpretability, and gives you hands-on
tools to peer inside GPT-2.

## 1. Once Upon a Time

### 1.1 It Was Only Neuroscience (McCulloch & Pitts, 1943)

In 1943, Warren McCulloch and Walter Pitts proposed the first formal model of a neuron,
framing the brain as a graph $G=(V, E)$ where every node $v_i \in V$ performs a single,
crisp calculation to decide its state.

The **state** $y_i$ of neuron $v_i$ is determined by the weighted sum of its inputs:
$$
y_i = \mathbf{1} \Biggl( \sum_j w_{ij} \cdot x_j \geq \theta_i \Biggr)
$$

where:
- $j$ indexes all neurons sending *binary* signals into $i$
- $w_{ij}$ is the synaptic weight from neuron $j$ to neuron $i$
- $\theta_i$ is the threshold, acting as a gatekeeper

If neuron $v_i$ fires, its state is $y_i = 1$; otherwise $y_i = 0$.

In 1943, weights were fixed: $+1$ for excitatory (encouraging a fire) or $-1$ for inhibitory
(preventing a fire). By tuning both $w_{ij}$ and $\theta_i$, this single computational cell
can model `AND` and `OR` gates, making it a universal computer.

### 1.2 From Logics to Energies: The Ising Model

For decades, the graph was viewed as a rigid "calculator." But in the 1970s, physics
entered the room. Researchers looked at the **Ising Model**, which describes how atoms
$i \in \{1, \ldots, n\}$ in a lattice influence each other's magnetic spin $s_i \in \{-1, +1\}$.

The system is governed by an **energy function**:
$$
E(s) = - \sum_{(i,j)} J_{ij} \cdot s_i \cdot s_j - \sum_i h_i \cdot s_i
$$

If atoms $i$ and $j$ have a positive coupling $J_{ij}$, they "want" to align to lower the
system's energy. This introduced the idea that **computation is an optimisation process** —
a search for the lowest energy state.

### 1.3 Hopfield Networks

John Hopfield (1982) observed that the Ising interaction structure could be **reinterpreted
as a computational system that stores memories**, rather than merely a model of magnetic
materials.

A **Hopfield network** consists of $n$ neurons with binary states $s_i \in \{-1, +1\}$.
The full configuration $s = (s_1, \ldots, s_n)$ represents the current pattern (e.g.,
a black-and-white image where $+1$ = white pixel, $-1$ = black pixel).

Neurons update **asynchronously**:
$$
s_i = \text{sign} \Biggl( \sum_{j=1}^n w_{ij} \cdot s_j \Biggr)
$$

The weight matrix $W = (w_{ij})$ is typically **symmetric** ($w_{ij} = w_{ji}$) and has
**no self-connections** ($w_{ii} = 0$).

Hopfield defined a global **energy function**:
$$
E(s) = -\frac{1}{2} \sum_{i,j} w_{ij} \cdot s_i \cdot s_j
$$

Each neuron update **never increases energy**: $E(s_{\text{new}}) \leq E(s_{\text{old}})$.
The network therefore converges to **local energy minima** — these are the **attractor states**
that represent stored memories.

> **Key insight:** we can *design* weights $W$ so that specific patterns become energy minima.
> If the network starts from a noisy version of a stored pattern, it will denoise it by
> rolling downhill in energy.

<img src="images/Gemini_Generated_Image_noedusnoedusnoed.png" width="800">

### 1.4 Boltzmann Machines

Building on Hopfield networks, Hinton & Sejnowski (1985) introduced **Boltzmann machines** —
a *stochastic*, energy-based network capable of *learning* probability distributions over data.

The energy function has the same form as the Hopfield network, but now with explicit **bias terms**:
$$
E(s) = -\sum_{i<j} w_{ij} \cdot s_i \cdot s_j - \sum_i b_i \cdot s_i
$$

Unlike Hopfield networks (deterministic updates), Boltzmann machines use **probabilistic updates**:
$$
P(s_i = 1 \mid s_{-i}) = \sigma\Biggl(\sum_j w_{ij} \cdot s_j + b_i\Biggr)
$$

This stochasticity allows the network to escape local minima.

| Feature | Hopfield Network | Boltzmann Machine |
|---------|-----------------|-------------------|
| Updates | Deterministic   | Probabilistic (temperature) |
| Weights | **Hand-designed** to store specific patterns | **Learned from data** |
| Minima  | Explicitly designed attractor states | Emergent from training |
| Output  | Stored memory recall | Generative model (new samples) |

> **📖 PML Ch. 3 §3.6 — Markov Random Fields and the Boltzmann Distribution**
>
> Chapter 3 (§3.6) introduces **Markov Random Fields (MRFs)** as *undirected* graphical models,
> contrasting with the directed Bayesian Networks studied earlier. Nodes are random variables;
> edges encode dependency. Conditional independence in an MRF is strikingly simple
> (Proposition 3.6.1): $A \perp\!\!\!\perp B \mid C$ if and only if **every path** from $A$ to $B$
> passes through $C$ — no need for the three-motif d-separation analysis required in BNs.
>
> The joint factorises over **maximal cliques** $\mathcal{C}$ of the graph (Def 3.6.2–3):
> $$p(x) = \frac{1}{Z} \prod_{C \in \mathcal{C}} \Psi_C(x_C), \qquad Z = \int \prod_C \Psi_C(x_C)\,dx_C$$
> where $\Psi_C \geq 0$ is a **potential function** (Def 3.6.4) and $Z$ is the **partition function**
> (Def 3.6.5) — typically intractable. To guarantee $\Psi_C \geq 0$, the standard choice is
> the **Boltzmann distribution**: $\Psi_C(x_C) = \exp(-E_C(x_C))$, which satisfies
> $\Psi_{C_1} \cdot \Psi_{C_2} = \exp(-E_1 - E_2)$ — products of potentials become sums of energies.
>
> The notes explicitly lists **Boltzmann Machines** and the **Ising Model** as canonical MRF examples
> (§3.6, final remark). The Boltzmann Machine you just read about has energy
> $E(s) = -\sum_{i<j} w_{ij} s_i s_j - \sum_i b_i s_i$, which is exactly the Ising-type quadratic
> energy over the pairwise cliques of a fully connected graph.
>
> **Why this matters for LLMs:** The softmax in every attention head is a Boltzmann distribution.
> The attention score $\exp(q_i^\top k_j / \sqrt{d})$ is a potential $\Psi(i,j) = \exp(-E_{ij})$
> with $E_{ij} = -q_i^\top k_j / \sqrt{d}$. The softmax denominator is exactly the partition
> function $Z$ — and unlike general MRFs, it is **tractable** because attention is computed
> one row at a time (one query against all keys), keeping the clique size constant.

### 1.5 Why Do We Care?

While Boltzmann machines operate on binary neurons with explicit energy functions, modern
neural networks generalise these ideas to *continuous activations, directed weights, and
high-dimensional computations*.

**The key conceptual bridge:**
- In Hopfield/Boltzmann networks, we can trace exactly *which weights* store *which memories*.
- In modern deep networks, the same mechanistic reasoning applies — we can identify *which
  circuits* (subsets of neurons, attention heads, MLP blocks) are responsible for *specific
  computations*.

This is the foundation of **mechanistic interpretability**: understanding neural networks by
reverse-engineering the algorithms implemented by their components.

## 2. Transformers

Transformers, the architecture behind modern LLMs like GPT-2, generalise the same principle
we saw in Hopfield and Boltzmann machines: complex computation emerges from the interactions
of many units through weighted connections.

**Graph perspective:**
- **Nodes:** vectors representing intermediate states — the residual stream, outputs of
  attention heads, MLP block outputs.
- **Edges:** parameterised transformations — linear projections, attention operations,
  layer normalisation steps.

Global behaviour (recalling a fact, predicting the next token, performing arithmetic)
emerges from the composition of these nodes and edges.

### The Residual Stream

A critical architectural feature is the **residual stream**: each attention head and MLP
block *adds* its output to a running vector (the residual stream) rather than replacing it.
This means:
1. Earlier layers' computations are preserved throughout the network.
2. Components can be thought of as independently contributing to the final prediction.
3. We can surgically **patch** (replace) the residual stream at any layer and position
   to trace where information lives.

> **📖 PML Ch. 3 §3.1–3.3 — Factorization Costs and Bayesian Networks**
>
> **§3.1.2 — Why factorizations matter.** Chapter 3 opens with a stark computational argument.
> Storing a full joint $p(x_1, \ldots, x_n)$ over binary variables costs $O(2^n)$ — infeasible
> for $n = 100$. But a **chain factorization** $p(x_n | x_{n-1}) \cdots p(x_2 | x_1) p(x_1)$
> with $k$-valued variables costs only $O(n k^2)$ — tractable. The key is finding which
> variables are *conditionally independent given their parents*, so we can drop them from the
> conditioning set. **Probabilistic Graphical Models encode exactly this sparse factorization.**
>
> **§3.2 — Bayesian Networks.** A BN is a DAG $G$ with nodes $\{x_1, \ldots, x_n\}$
> and a directed edge $x_i \to x_j$ whenever $x_i \in pa_j$ (the parents of $x_j$). The joint
> factorises as $p(x_1, \ldots, x_n) = \prod_k p(x_k \mid pa_k)$.
>
> **The Transformer is a BN in disguise.** The autoregressive factorisation
> $p(w_1, \ldots, w_T) = \prod_t p(w_t \mid w_1, \ldots, w_{t-1})$ is exactly a BN
> where each token's parents are all preceding tokens. Inside the network, the residual stream
> at layer $\ell$, position $p$ has parents: the residual at layer $\ell{-}1$ (all positions,
> via attention) and the MLP output at layer $\ell$.
>
> **§3.3.1 — Ancestral sampling = autoregressive generation.** Chapter 3 describes
> *ancestral sampling*: sample root nodes first, then each child using already-sampled parents.
> This is exactly how GPT-2 generates text: sample $w_1$ from $p(w_1)$, then $w_2$ from
> $p(w_2 | w_1)$, and so on — each token is a child whose parents are all previous tokens.
>
> **§3.3.2 — Three reasoning modes.** Chapter 3 identifies three forms of inference in a BN:
>
> | Reasoning mode | Direction | BN example | Transformer equivalent |
> |---|---|---|---|
> | **Causal** | top → bottom | observe Difficulty → predict Grade | observe country token → predict capital |
> | **Evidential** | bottom → top | observe Grade → infer Difficulty | observe output logit → trace back to source layer |
> | **Intercausal** | both directions | Grade observed, D and I interact | activation patching propagates both forward and backward |
>
> Activation patching is fundamentally **intercausal reasoning**: we intervene on an
> intermediate node (the residual stream at layer $L$) and observe the effect both upstream
> (which layers sourced the information) and downstream (which components use it).

### 2.1 LLM Architecture: A Bird's-Eye View

Before diving into mechanistic experiments, it helps to have a concrete picture of how a Transformer LLM works end-to-end.

```
Input text: "The capital of France is"
        │
        ▼
┌──────────────────────────────────────────────────────────────┐
│  TOKENISER                                                   │
│  Splits text into sub-word tokens                            │
│  "The", " capital", " of", " France", " is"                  │
└──────────────────────────────┬───────────────────────────────┘
                               │  integer IDs
                               ▼
┌──────────────────────────────────────────────────────────────┐
│  EMBEDDING LAYER                                             │
│  Each token ID → a dense vector of size d_model (768 in     │
│  GPT-2 Small). These vectors are the starting point of the  │
│  residual stream.                                            │
└──────────────────────────────┬───────────────────────────────┘
                               │  shape: [seq_len, d_model]
                               ▼
┌──────────────────────────────────────────────────────────────┐
│  TRANSFORMER BLOCKS  ×12  (for GPT-2 Small)                  │
│                                                              │
│  Each block contains two sub-modules:                        │
│                                                              │
│  ┌─────────────────────────────────────────────────────┐    │
│  │  MULTI-HEAD ATTENTION  (12 heads)                   │    │
│  │  Each head independently:                           │    │
│  │   1. Projects every token into Query (Q), Key (K),  │    │
│  │      Value (V) vectors                              │    │
│  │   2. Computes attention scores: softmax(QKᵀ / √d)  │    │
│  │   3. Reads a weighted mix of V vectors              │    │
│  │  Heads specialise: some track syntax, some copy    │    │
│  │  tokens, some look up factual associations.        │    │
│  └──────────────────────────┬──────────────────────────┘    │
│                             │  added to residual stream      │
│  ┌──────────────────────────▼──────────────────────────┐    │
│  │  MLP (feed-forward network, 2 layers)               │    │
│  │  Acts as a key-value memory bank:                   │    │
│  │   • First layer: "pattern matching" (what do I see?)│    │
│  │   • Second layer: "value read-out" (what to output) │    │
│  │  Stores world knowledge as patterns in its weights. │    │
│  └──────────────────────────┬──────────────────────────┘    │
│                             │  added to residual stream      │
└─────────────────────────────┼────────────────────────────────┘
                              │  repeated 12 times
                              ▼
┌──────────────────────────────────────────────────────────────┐
│  UNEMBEDDING LAYER                                           │
│  Projects the final residual stream vector at the last       │
│  token position into a probability distribution over the     │
│  entire vocabulary (~50,000 tokens for GPT-2).               │
│  The top token is the model's prediction.                    │
└──────────────────────────────────────────────────────────────┘
        │
        ▼
Output: " Paris"
```

**Key design choice — the residual connection:** every sub-module *adds* its output to the running vector rather than replacing it. This means the vector is never erased — only enriched.

---

### 2.2 The Residual Stream as Plumbing 🔧

Think of the residual stream as a **pipe running through the entire network**, one pipe per token position.

- At the **input end**, each pipe is filled with the raw embedding of that token.
- At each layer, the attention heads and the MLP **deposit small updates** into the pipe — they don't drain it, they add to it.
- By the **output end**, the pipe contains a rich mixture of all the contributions from all the layers that came before.

```
Token position:   [BOS]    "The"  " capital"   " of"   " France"   " is"
                    │         │        │          │         │          │
                  pipe      pipe     pipe       pipe      pipe       pipe
                    │         │        │          │         │          │
Layer 0  Attn ──adds──      ...      ...        ...    ←─reads─     adds
Layer 0  MLP  ──adds──      ...      ...        ...      adds       adds
Layer 1  Attn                                           ←─reads─     adds
...
Layer 9  Attn  ←─ CAUSAL BOTTLENECK: head L9_H8 reads " France" pipe, writes to " is" pipe
...
Layer 11 MLP                                                         adds
                                                                      │
                                                                  Unembedding
                                                                      │
                                                                   "Paris"
```

**Why is this useful for mechanistic interpretability?**

Because the residual stream is the *only* medium through which components communicate, we can **intercept it at any point**. By reading (or replacing) the vector in a specific pipe at a specific layer, we can measure exactly what information each component contributed — which is precisely what activation patching does.


<img src="images/Gemini_Generated_Image_khuoh3khuoh3khuo.png" width="800">

<img src="images/unnamed.jpg" width="800">

## 3. Experimental Setup

### 3.1 Install Dependencies

`transformer_lens` is a library specifically designed for mechanistic interpretability.
It wraps HuggingFace models and exposes **hooks** at every internal computation,
allowing us to read, modify, or replace any activation tensor.

In [ ]:
!pip uninstall -y transformer_lens numpy opencv-python opencv-python-headless opencv-contrib-python jax jaxlib cupy-cuda12x shap rasterio pytensor
!pip install numpy==1.26.4
!pip install transformer_lens

### 3.2 Load GPT-2 Small

We load **GPT-2 Small** (124M parameters, 12 layers, 12 attention heads per layer,
hidden dimension 768). We then define a **counterfactual pair** of prompts:

| Prompt | Expected completion |
|--------|--------------------|
| `"The capital of France is"` | `" Paris"` |
| `"The capital of Italy is"` | `" Rome"` |

Our **metric** throughout this notebook is the **logit difference**:
$$
\text{score} = \text{logit}_{\text{Paris}} - \text{logit}_{\text{Rome}}
$$

- A **high positive score** means the model strongly prefers "Paris" (correct for France)
- A **strongly negative score** means the model strongly prefers "Rome" (correct for Italy)
- A score **near zero** means the model is uncertain between the two

In [ ]:
import torch
from transformer_lens import HookedTransformer

# Fix the random seed for reproducibility
torch.manual_seed(1337)

# ── Load the model ──────────────────────────────────────────────────────────
# HookedTransformer wraps GPT-2 and exposes internal activations via hooks.
# This is a 124M parameter model with:
#   - 12 transformer layers
#   - 12 attention heads per layer
#   - hidden dimension of 768
print("Loading GPT-2 Small...")
model = HookedTransformer.from_pretrained("gpt2-small")
print(f"Model loaded: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")

# ── Define clean and corrupted prompts ─────────────────────────────────────
# 'Clean' prompt: the factual one we want to understand
# 'Corrupted' prompt: the counterfactual used to measure causal influence
prompt_clean = "The capital of France is"
prompt_corr  = "The capital of Italy is"

# Tokenise both prompts
# Note: both prompts have the same structure and length, differing only at the
# country token ("France" vs "Italy"). This controlled difference is essential
# for a valid causal experiment.
tokens_clean = model.to_tokens(prompt_clean)
tokens_corr  = model.to_tokens(prompt_corr)

print(f"Clean tokens: {model.to_str_tokens(tokens_clean)}")
print(f"Corrupted tokens: {model.to_str_tokens(tokens_corr)}")

# ── Get the token IDs for our target words ──────────────────────────────────
# These are the two vocabulary items we will track throughout the experiment.
token_paris = model.to_single_token(" Paris")  # note the leading space!
token_rome  = model.to_single_token(" Rome")
print(f"Token IDs — Paris: {token_paris}, Rome: {token_rome}")

### 3.3 Baseline Logit Differences

We define a helper function to compute our key metric, then run both prompts through
the model to establish baseline scores.

`model.run_with_cache()` returns:
1. The final logits (shape: `[batch, seq_len, vocab_size]`)
2. A **cache** dictionary containing every intermediate activation, indexed by hook name
   (e.g. `"blocks.5.hook_resid_post"`, `"blocks.3.attn.hook_pattern"`, etc.)

In [ ]:
def get_logit_diff(logits):
    """
    Compute the logit difference (Paris - Rome) at the final token position.
    
    This is our main scalar metric throughout the notebook:
      > 0  → model prefers 'Paris'
      < 0  → model prefers 'Rome'
      ≈ 0  → model is uncertain between them
    
    Args:
        logits: tensor of shape [batch, seq_len, vocab_size]
    Returns:
        float: logit(Paris) - logit(Rome) at the last token position
    """
    # We look at the LAST token position because GPT-2 predicts the next token
    # from the last position in the sequence.
    final_logits = logits[0, -1, :]  # shape: [vocab_size]
    return (final_logits[token_paris] - final_logits[token_rome]).item()


# ── Run both prompts and cache ALL internal activations ─────────────────────
# cache_clean contains the activations produced by the factual prompt
# cache_corr  contains the activations produced by the counterfactual prompt
logits_clean, cache_clean = model.run_with_cache(tokens_clean)
logits_corr,  cache_corr  = model.run_with_cache(tokens_corr)

# ── Establish baselines ─────────────────────────────────────────────────────
clean_diff = get_logit_diff(logits_clean)
corr_diff  = get_logit_diff(logits_corr)

print("=" * 50)
print("BASELINE LOGIT DIFFERENCES")
print("=" * 50)
print(f"Clean  ('The capital of France is'): {clean_diff:+.4f}")
print(f"  → Model strongly prefers 'Paris' ✓")
print()
print(f"Corrupted ('The capital of Italy is'): {corr_diff:+.4f}")
print(f"  → Model strongly prefers 'Rome' ✓")
print()
print("The gap between these two values defines the full range of our experiments.")
print(f"  Range: {corr_diff:.2f}  ←──────────── 0 ────────────→  {clean_diff:.2f}")

<img src="images/Gemini_Generated_Image_7c9vlb7c9vlb7c9v.png" width="800">

## 4. Causal Tracing

> **📖 PML Ch. 3 §3.4 — Conditional Independence and d-Separation in Bayesian Networks**
>
> Chapter 3 (§3.4, Def 3.4.1) defines conditional independence precisely:
> $a \perp\!\!\!\perp b \mid c$ iff $p(a \mid b, c) = p(a \mid c)$, or equivalently
> $p(a, b \mid c) = p(a \mid c)\, p(b \mid c)$.
>
> Three **elementary motifs** determine whether a path between two nodes is blocked:
>
> | Motif | Structure | Key result |
> |---|---|---|
> | **Tail-to-tail** (§3.4.1) | $a \leftarrow c \rightarrow b$ | $a \not\perp b$ but $a \perp b \mid c$ — observing the *common cause* blocks the path |
> | **Head-to-tail** (§3.4.2) | $a \rightarrow c \rightarrow b$ | $a \not\perp b$ but $a \perp b \mid c$ — observing the *mediator* blocks the path |
> | **Head-to-head** (§3.4.3) | $a \rightarrow c \leftarrow b$ | $a \perp b$ but $a \not\perp b \mid c$ — observing the *collider* **opens** the path ("explaining away"); also $a \not\perp b \mid d$ for any descendant $d$ of $c$ |
>
> **Global criterion (Proposition 3.4.1):** $A \perp\!\!\!\perp B \mid C$ iff **every path** from
> $A$ to $B$ is blocked by $C$ under the above rules. This is called **d-separation**.
>
> **Activation patching is a computational d-separation test.**
>
> - $X$ = output logit (Paris vs Rome)  
> - $Y$ = activations from the corrupted run (Italy)  
> - $Z$ = residual stream at layer $L$, position $p$
>
> Patching layer $L$ at position $p$ with clean activations asks:
> *"Is $Z$ a sufficient **blocking set** — does $X \perp\!\!\!\perp Y \mid Z$?"*
>
> High recovery = yes, $Z$ d-separates $X$ from $Y$: this $(L, p)$ cell lies on every
> information path from the corrupted input to the output.
> The full **heatmap** in Section 4.3 is a simultaneous d-separation test over all $(L, p)$ pairs.

### 4.1 Conditional Independence and Activation Patching

To locate *where* information is processed inside the model, we use a technique called
**activation patching** (also called causal tracing or causal mediation analysis).

The key idea comes from probability theory. **Conditional independence** states:
$$
X \perp Y \mid Z
$$
"Once you know $Z$, knowing $Y$ gives you no extra information about $X$."

In our experiment:
- **$X$** = the predicted next token (Paris or Rome)
- **$Y$** = hidden states from an earlier, *corrupted* input ("The capital of Italy is")
- **$Z$** = the residual stream at some layer $L$

$$
\underbrace{\text{next token}}_{X} \perp \underbrace{\text{corrupted activations before layer } L}_{Y} \mid \underbrace{\text{residual stream at layer } L}_{Z}
$$

**Intuition:** the model only "cares" about the current hidden state at layer $L$.
If we *replace* the residual stream at layer $L$ in the corrupted run with the one from
the clean run, we're asking: *"Is layer $L$ sufficient to restore the correct prediction?"*

**Procedure:**
1. Run the *corrupted* input (Italy) → model predicts Rome.
2. At each layer $L$, **patch** the residual stream at the last token position with the
   value from the *clean* run (France).
3. Let the rest of the network run normally.
4. Measure whether the model now predicts Paris instead of Rome.
5. A large jump toward Paris means layer $L$ carries crucial information about the country.

> **📖 PML Ch. 3 §3.4.1–3.4.3 — The Three CI Motifs in the Transformer**
>
> The three motifs from Chapter 3 appear concretely in the Transformer's information flow:
>
> **Tail-to-tail (common cause):** The country token `"France"` is a common cause for
> both the attention heads that read it *and* the MLP layers that retrieve its capital.
> Unconditionally, these components are correlated (they all respond to France).
> Once we condition on (patch) the `"France"` residual stream in early layers, the
> downstream attention heads and MLPs become independent — the common cause is fixed.
>
> **Head-to-tail (chain / mediator):** The information flows
> `country token (L0–8) → last-token residual (L9) → output`.
> This is a head-to-tail chain with the L9 residual as mediator.
> Conditioning on (patching) L9 blocks all upstream influence — which is why patching
> layer 11 gives 100% recovery: the final residual is a sufficient mediator for the whole path.
>
> **Head-to-head (collider / explaining away):** At layer 9, the last-token (`"is"`)
> residual stream receives writes from **both** the attention head L9_H8 (reading France)
> **and** earlier MLP layers (background knowledge). This position is a collider.
> Once we observe (patch) this residual, the two parent streams become dependent:
> ablating L9_H8 causes the MLP contribution to "over-compensate", and vice versa.
> This explains why ablating some heads can *increase* the logit difference — the
> collider's explaining-away effect redistributes credit between parents.

<img src="images/Gemini_Generated_Image_x7w6s4x7w6s4x7w6.png" width="800">

### 4.2 Layer-by-Layer Scan

In [ ]:
print("Layer-by-Layer Causal Tracing")
print("-" * 40)

patching_results = []  # stores logit diff after patching each layer

for layer in range(model.cfg.n_layers):

    # ── Choose which activation to patch ────────────────────────────────────
    # 'hook_resid_post' is the residual stream AFTER the full transformer block
    # (i.e. after attention + MLP + layer norm) at the given layer.
    hook_name = f"blocks.{layer}.hook_resid_post"

    # ── Define the patching function ────────────────────────────────────────
    # This hook runs DURING a forward pass on the corrupted input.
    # It replaces the activation at the LAST token position with the
    # corresponding activation from the clean run.
    # We only patch the last token because that's where the prediction happens.
    def patch_hook(corrupted_resid, hook):
        patched_resid = corrupted_resid.clone()
        # Transplant clean activations at the last token position only
        patched_resid[:, -1, :] = cache_clean[hook.name][:, -1, :]
        return patched_resid

    # ── Run the corrupted input with the patch active ────────────────────────
    patched_logits = model.run_with_hooks(
        tokens_corr,
        fwd_hooks=[(hook_name, patch_hook)]
    )

    patched_diff = get_logit_diff(patched_logits)
    patching_results.append(patched_diff)

    # Indicate how close we got to the clean baseline
    recovery_pct = (patched_diff - corr_diff) / (clean_diff - corr_diff) * 100
    bar = "█" * max(0, int(recovery_pct / 5))
    print(f"Layer {layer:02d} | diff={patched_diff:+.4f} | recovery={recovery_pct:5.1f}% |{bar}")

print()
print(f"Reference — Corrupted baseline: {corr_diff:+.4f} (0% recovery)")
print(f"Reference — Clean baseline:     {clean_diff:+.4f} (100% recovery)")

**Reading the results:**

- Layers 0–8 show low recovery: patching the last-token residual stream at these early/mid
  layers is *not* enough to restore the correct prediction. The country-specific information
  has not yet been concentrated there.
- Layer 9–10: a **sharp jump** in recovery. This is the critical transition where the model
  moves from *storing* the subject information at the country token to *preparing* the answer
  at the last token position.
- Layer 11: patching the final-layer residual completely restores the clean prediction
  (100% recovery) — because by then, the decision is already encoded in that vector.

### 4.3 Full Causal Tracing Heatmap (All Layers × All Token Positions)

The layer-by-layer scan only patched the *last* token position. But information might be
stored at *other* positions (e.g. at the country token itself).

We now perform a **full 2D causal trace**, patching every combination of:
- Layer $L \in \{0, 1, \ldots, 11\}$
- Token position $p \in \{0, 1, \ldots, T-1\}$

This produces a **heatmap** showing where in (layer, position) space the relevant
information is stored.

> **📖 PML Ch. 3 §3.4.4 Proposition 3.4.1 — D-Separation as a Global Visual Test**
>
> **Proposition 3.4.1 (Ch. 3):** Subsets $A$ and $B$ are conditionally independent given
> $C$ ($A \perp\!\!\!\perp B \mid C$) if and only if **every path** from any node in $A$
> to any node in $B$ is blocked by some node in $C$, following the three motif rules.
>
> The full $(\text{layer} \times \text{position})$ heatmap below is a direct visual
> instantiation of this proposition:
>
> - **Each cell $(\ell, p)$** tests a candidate blocking set $C = \{$residual at layer $\ell$, position $p\}$.
> - A **warm (red) cell** means this single-node set is sufficient to d-separate the
>   corrupted input $Y$ from the output $X$ — all information paths pass through it.
> - A **cold (blue) cell** means this node is off the information path: conditioning on it
>   leaves the corrupted influence intact.
>
> Reading the heatmap through Ch. 3's lens:
>
> | Region | Ch. 3 explanation |
> |---|---|
> | `"France"` column, layers 0–8 warm | Country identity is stored *at the subject token*; conditioning there blocks the tail-to-tail / head-to-tail path from country to output |
> | `"France"` column, layer 9 sudden cold | Information has been **moved away** from the subject token — conditioning there is no longer sufficient |
> | `"is"` column, layers 9–11 warming | The fact is now concentrated at the prediction position; this new node becomes a valid blocking set |
> | Most other columns cold throughout | These token positions carry no country-specific information — conditioning on them never blocks the relevant paths |

In [ ]:
import numpy as np
import plotly.graph_objects as go

all_layers = list(range(model.cfg.n_layers))
# Get human-readable token strings for the corrupted prompt (used as x-axis labels)
str_tokens = model.to_str_tokens(tokens_corr)

# heatmap_data[layer][position] = logit_diff after patching that (layer, position) pair
heatmap_data = []

print(f"Running full causal trace ({len(all_layers)} layers × {len(str_tokens)} positions)...")

for layer in all_layers:
    layer_results = []
    hook_name = f"blocks.{layer}.hook_resid_post"

    for pos in range(len(str_tokens)):

        # ── Patch a single (layer, position) cell ───────────────────────────
        # We use a default-argument trick (pos=pos) to capture the current
        # value of `pos` in the closure — without this, all closures would
        # share the same reference to the loop variable.
        def patch_position_hook(corrupted_resid, hook, pos=pos):
            patched_resid = corrupted_resid.clone()
            patched_resid[:, pos, :] = cache_clean[hook.name][:, pos, :]
            return patched_resid

        patched_logits = model.run_with_hooks(
            tokens_corr,
            fwd_hooks=[(hook_name, patch_position_hook)]
        )

        layer_results.append(get_logit_diff(patched_logits))

    heatmap_data.append(layer_results)

print("Done! Rendering heatmap...")

# ── Visualise ───────────────────────────────────────────────────────────────
# Colour scale: red = high (Paris preferred), blue = low (Rome preferred)
# zmid=0 centres the colour scale so white = neutral
fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=str_tokens,
    y=[f"L{l}" for l in all_layers],
    colorscale="RdBu_r",
    zmid=0,
    colorbar=dict(title="Logit Diff<br>(Paris − Rome)")
))

fig.update_layout(
    title="Full Causal Trace: Where is 'France → Paris' Knowledge Processed?",
    xaxis_title="Input Token (Position)",
    yaxis_title="Layer",
    font=dict(size=13)
)
fig.show()

**How to read this heatmap:**

| Region | Colour | Interpretation |
|--------|--------|----------------|
| `Italy` column, layers 0–8 | Warm (red/orange) | Country identity is **stored at the subject token** in early layers. Patching here restores Paris. |
| `Italy` column, layer 9 | Sudden shift to cold | **Causal bottleneck**: information is being moved *away* from the subject token. |
| `is` column, layers 9–11 | Warming up | The answer is being **collected at the last token** in preparation for output. |
| `is` column, layer 11 | Fully warm | By the final layer, **all necessary information is concentrated** at the prediction position. |

Most columns other than `Italy` and `is` stay cold throughout — they carry little
country-specific information relevant to this prediction.

This picture reveals a two-phase process:
1. **Early/mid layers (0–8):** Knowledge retrieval — the model reads "Italy" and fetches
   associated attributes (capital city) at the subject token position.
2. **Late layers (9–11):** Information routing — the retrieved fact is moved to the last
   token position where the vocabulary unembedding will produce the output.

## 5. Component Analysis: Attention Heads vs. MLP Layers

The causal heatmap tells us *where and when* information flows. Now we drill deeper
to ask: **which specific components** (attention heads and MLP layers) are responsible?

| Component | Hypothesised role |
|-----------|------------------|
| **Attention heads** | *Transport system* — move information between token positions |
| **MLP layers** | *Memory bank* — store and retrieve factual associations |

We test these roles with two complementary experiments:
1. **Attention head mapping:** which heads move information from the subject token to the
   last token?
2. **MLP ablation:** which MLP layers contribute to the correct prediction?

### 5.1 Attention Head Mapping

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

n_layers = model.cfg.n_layers  # 12
n_heads  = model.cfg.n_heads   # 12

# ── Measure how much each head attends from 'is' → 'France' ─────────────────
# We're interested in heads that look BACK at the subject token from the FINAL
# token position, since those are most likely to move subject information forward.
#
# hook_pattern shape: [batch, head_index, destination_token, source_token]
#   - destination = -1  (the last token, 'is')
#   - source      = -2  (the second-to-last token, 'France')
attn_to_subject = torch.zeros((n_layers, n_heads))

for l in range(n_layers):
    pattern = cache_clean[f"blocks.{l}.attn.hook_pattern"][0]  # [n_heads, seq, seq]
    # Attention weight: how much does 'is' (pos -1) attend to 'France' (pos -2)?
    attn_to_subject[l] = pattern[:, -1, -2].detach().cpu()

# ── Visualise ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    attn_to_subject.numpy(),
    cmap="Reds",
    annot=True,
    fmt=".2f",
    ax=ax,
    cbar_kws={"label": "Attention weight: 'is' → 'France'"}
)
ax.set_title("Attention Head Map: Which Heads Route Information from Subject to Prediction?",
             fontsize=13, pad=12)
ax.set_xlabel("Head Index", fontsize=12)
ax.set_ylabel("Layer", fontsize=12)
ax.invert_yaxis()  # Layer 0 at bottom
plt.tight_layout()
plt.show()

# ── Find candidate 'mover' heads above a threshold ──────────────────────────
# A head with high attention weight from 'is' to 'France' is a CANDIDATE mover.
# Note: high attention ≠ causally important — we test causality next.
threshold = 0.2
mover_heads = torch.nonzero(attn_to_subject > threshold, as_tuple=False)
print(f"Candidate mover heads (attention weight > {threshold}):")
for l, h in mover_heads:
    w = attn_to_subject[l, h].item()
    print(f"  Layer {l:2d}, Head {h:2d}  (weight = {w:.3f})")

### 5.2 Causal Ablation of Attention Heads

High attention weight is a **correlational** signal — the head *looks at* the subject.
But does it *causally* affect the prediction?

We test this by **ablating** each candidate head: we zero out its contribution to the
residual stream and measure how much the logit difference drops.

- **Large drop** → this head causally contributes to the correct answer
- **Small/no drop** → this head is attending but not doing important work
- **Negative drop** (improvement) → this head was actually *hurting* the prediction
  (e.g. by promoting a competing token)

We ablate by zeroing `hook_z`, the per-head output tensor before it is projected and
added to the residual stream. Shape: `[batch, position, n_heads, d_head]`.

<img src="images/Gemini_Generated_Image_y74iuoy74iuoy74i.png" width="800">

In [ ]:
print("Targeted Head Ablation Experiment")
print("-" * 50)
print(f"Clean baseline (intact model): {clean_diff:+.4f}")
print()

def make_z_ablation_hook(head_index):
    """
    Returns a hook that zeros out a specific attention head's output.
    
    We use a factory function so each hook correctly captures its own
    head_index (avoiding Python closure gotchas in loops).
    
    Args:
        head_index: which head to ablate (0-indexed)
    Returns:
        hook function suitable for model.run_with_hooks()
    """
    def ablation_hook(z, hook):
        # z shape: [batch, position, n_heads, d_head]
        ablated_z = z.clone()
        ablated_z[:, :, head_index, :] = 0.0  # zero out this head at all positions
        return ablated_z
    return ablation_hook


# ── Ablate each candidate head individually ──────────────────────────────────
ablation_results = []
for l, h in mover_heads:
    hook_name = f"blocks.{l}.attn.hook_z"
    logits_ablated = model.run_with_hooks(
        tokens_clean,
        fwd_hooks=[(hook_name, make_z_ablation_hook(h))]
    )
    diff = get_logit_diff(logits_ablated)
    drop = clean_diff - diff
    ablation_results.append((l.item(), h.item(), diff, drop))
    # Classify the head's role
    role = "■ IMPORTANT" if drop > 0.3 else ("○ minor" if drop > 0 else "▼ interfering")
    print(f"  L{l:02d}_H{h:02d}: diff={diff:+.4f}  drop={drop:+.4f}  {role}")

# ── Ablate ALL candidate heads simultaneously ────────────────────────────────
# This tells us the combined causal footprint of all mover heads.
print()
all_hooks = [
    (f"blocks.{l}.attn.hook_z", make_z_ablation_hook(h))
    for l, h in mover_heads
]
logits_all_ablated = model.run_with_hooks(tokens_clean, fwd_hooks=all_hooks)
diff_all = get_logit_diff(logits_all_ablated)
total_drop = clean_diff - diff_all
pct_explained = total_drop / (clean_diff - corr_diff) * 100
print(f"All mover heads ablated: diff={diff_all:+.4f}")
print(f"  Total drop:       {total_drop:+.4f}")
print(f"  % of full range:  {pct_explained:.1f}%")
print(f"  → Attention heads account for ~{pct_explained:.0f}% of the factual prediction;")
print(f"    the rest comes from MLP layers.")

### 5.3 MLP Layer Ablation

MLP layers in transformers act as **key-value memories** (Mor Geva et al., 2021): the first
linear layer activates "keys" that match patterns, and the second layer reads out
corresponding "values" (factual associations).

We ablate each MLP layer by replacing its entire output with zeros, then measure the
effect on the logit difference.

<img src="images/Gemini_Generated_Image_gqkqv8gqkqv8gqkq.png" width="800">

In [ ]:
print("MLP Layer Ablation Experiment")
print("-" * 50)
print(f"Clean baseline: {clean_diff:+.4f}")
print()

def mlp_ablation_hook(mlp_out, hook):
    """
    Zero out the entire output of an MLP layer.
    This removes the MLP's contribution to the residual stream for this forward pass.
    """
    return torch.zeros_like(mlp_out)


mlp_drops = []

for layer in range(model.cfg.n_layers):
    hook_name = f"blocks.{layer}.hook_mlp_out"

    logits_ablated = model.run_with_hooks(
        tokens_clean,
        fwd_hooks=[(hook_name, mlp_ablation_hook)]
    )

    diff = get_logit_diff(logits_ablated)
    drop = clean_diff - diff
    mlp_drops.append(drop)

    role = "■ IMPORTANT" if drop > 0.3 else ("▼ suppresses" if drop < -0.3 else "○ minor")
    print(f"  MLP Layer {layer:02d}: diff={diff:+.4f}  drop={drop:+.4f}  {role}")

print()
print("Interpretation guide:")
print("  Positive drop → MLP supports 'Paris' (removing it hurts)")
print("  Negative drop → MLP suppresses 'Paris' (removing it helps!)")
print("  Some MLPs actively compete against the correct answer, explaining")
print("  why the network needs 'balancing' circuits.")

**Key observations:**

- **MLP 1** consistently appears as an important memory layer across different countries
  (see the multi-country analysis below) — it likely performs early subject processing.
- **MLP 8** is important for France/Germany specifically — suggesting it stores European
  capital city associations.
- Several MLPs show **negative drops** (removing them *increases* the correct logit).
  This means those layers are actively *suppressing* Paris, perhaps because they're
  generating competing predictions. Neural networks are full of such "polyphonic" computations.

## 6. Finding Circuits (Markov Blankets)

> **📖 PML Ch. 3 §3.4.5 + §3.6.1 — Markov Blankets in Bayesian Networks and MRFs**
>
> Chapter 3 defines the Markov blanket formally in two settings, and they differ:
>
> **§3.4.5 — Markov Blanket in a Bayesian Network.**
> Conditioning $x_i$ on all other variables $x_{-i}$, the terms in the BN factorisation
> that do not involve $x_i$ cancel in numerator and denominator. What remains is:
> $$x_i \perp\!\!\!\perp x_{-i} \setminus \text{MB}(x_i) \mid \text{MB}(x_i)$$
> where $\text{MB}(x_i) = \underbrace{pa_i}_{\text{parents}} \cup \underbrace{\{x_j : x_i \in pa_j\}}_{\text{children}} \cup \underbrace{\bigcup_{x_j : x_i \in pa_j} pa_j \setminus \{x_i\}}_{\text{co-parents}}$
>
> In the **Transformer-as-DAG** view, the Markov blanket of the output node (final logit)
> is precisely the **minimal circuit**: the smallest set of attention heads and MLP layers
> that, once fixed, renders all other components irrelevant to the prediction.
>
> **§3.6.1 — Markov Blanket in a Markov Random Field.**
> In an MRF the definition is much simpler (Def 3.6.1):
> $\text{MB}(x_i) = \{$**neighbours** of $x_i$ in the graph$\}$.
> Conditional independence is also simpler than in BNs (Prop 3.6.1):
> $A \perp\!\!\!\perp B \mid C$ iff all paths from $A$ to $B$ pass through $C$ — no
> d-separation motif analysis needed, just topological path-blocking.
>
> | Concept | BN (directed, §3.4.5) | MRF (undirected, §3.6.1) |
> |---|---|---|
> | Markov blanket of $x_i$ | parents + children + co-parents | neighbours only |
> | CI criterion | d-separation (3 motifs) | simple path-separation |
> | Factorisation unit | conditional $p(x_k \mid pa_k)$ | clique potential $\Psi_C(x_C)$ |
> | Partition function | 1 (always normalised) | $Z$ — typically intractable |
>
> The **circuit-as-Markov-blanket** analogy holds in both frameworks:
> - In the BN view (layer-by-layer DAG): the circuit is the minimal *d-separating* set
>   between the input tokens and the output logit.
> - In the MRF view (attention as undirected graph): the circuit is the minimal
>   *clique cover* — the set of attention-head cliques and MLP potentials whose
>   product reproduces the output distribution.

When we search for a "circuit" inside a neural network, we're hunting for a statistical
boundary known in causal inference as a **Markov blanket**.

In a probabilistic graph, the **Markov blanket** of a target node $X$ is the minimal set
of nodes $Z$ such that:
$$
X \perp \text{(everything else)} \mid Z
$$

Applied to neural networks: a circuit is a **minimal subset of components** that is
sufficient to reproduce the model's behaviour on a given task. Knowing the circuit's
state makes all other model components irrelevant.

We construct circuits by:
1. **Keeping** the most causally important heads and MLP layers (found above).
2. **Zeroing** everything else.
3. **Measuring** whether the reduced circuit still predicts Paris.

<img src="images/Gemini_Generated_Image_seqwtdseqwtdseqw.png" width="800">

### 6.1 Minimal Circuit

In [ ]:
print("Minimal Circuit Experiment")
print("-" * 50)

# ── Define the minimal circuit ───────────────────────────────────────────────
# Based on the ablation experiments, we keep only the most impactful components:
#   - Attention head L2_H9  (drop ~0.39 in ablation)
#   - Attention head L3_H6  (drop ~0.49 in ablation)
#   - Attention head L9_H8  (drop ~0.68 in ablation)
#   - MLP layers 1 and 8    (both showed significant drops)
#
# Everything else is zeroed out.
important_heads_minimal = {
    2: [9],   # L2_H9
    3: [6],   # L3_H6
    9: [8],   # L9_H8
}
important_mlps_minimal = [1, 8]

print(f"Circuit components:")
print(f"  Attention heads: {important_heads_minimal}")
print(f"  MLP layers:      {important_mlps_minimal}")
print(f"  Clean baseline:  {clean_diff:+.4f}")


def make_head_mask_hook(important_heads_dict):
    """
    Returns a hook that zeros out all attention heads EXCEPT those listed in
    important_heads_dict[layer].
    
    Args:
        important_heads_dict: {layer_int: [head_int, ...]} — heads to KEEP
    """
    def hook(z, hook):
        layer = int(hook.name.split(".")[1])
        masked = z.clone()
        allowed = important_heads_dict.get(layer, [])  # empty → zero all heads
        for h in range(z.shape[2]):
            if h not in allowed:
                masked[:, :, h, :] = 0.0
        return masked
    return hook


def make_mlp_mask_hook(important_mlp_layers):
    """
    Returns a hook that zeros out all MLP outputs EXCEPT those at the specified layers.
    
    Args:
        important_mlp_layers: list of layer indices to KEEP
    """
    def hook(mlp_out, hook):
        layer = int(hook.name.split(".")[1])
        if layer in important_mlp_layers:
            return mlp_out  # pass through unchanged
        else:
            return torch.zeros_like(mlp_out)  # zero out
    return hook


# ── Build the hook list ──────────────────────────────────────────────────────
head_hook = make_head_mask_hook(important_heads_minimal)
mlp_hook  = make_mlp_mask_hook(important_mlps_minimal)

hooks_minimal = [
    (f"blocks.{l}.attn.hook_z",    head_hook)
    for l in range(model.cfg.n_layers)
] + [
    (f"blocks.{l}.hook_mlp_out",   mlp_hook)
    for l in range(model.cfg.n_layers)
]

logits_minimal = model.run_with_hooks(tokens_clean, fwd_hooks=hooks_minimal)
diff_minimal = get_logit_diff(logits_minimal)
pct_minimal = (diff_minimal - corr_diff) / (clean_diff - corr_diff) * 100

print(f"  Minimal circuit: {diff_minimal:+.4f}  ({pct_minimal:.1f}% of full prediction)")

### 6.2 Expanded Circuit

In [ ]:
print("Expanded Circuit Experiment")
print("-" * 50)

# ── Add more components to capture more of the prediction ───────────────────
# We include secondary mover heads (L0_H6, L1_H10) and additional MLP layers
# (2 and 3) that showed non-trivial ablation effects.
important_heads_expanded = {
    0: [6],    # L0_H6  — early mover
    1: [10],   # L1_H10 — early mover
    2: [9],    # L2_H9  — key mover
    3: [6],    # L3_H6  — key mover
    9: [8],    # L9_H8  — key mover
}
important_mlps_expanded = [1, 2, 3, 8]

print(f"Circuit components:")
print(f"  Attention heads: {important_heads_expanded}")
print(f"  MLP layers:      {important_mlps_expanded}")
print(f"  Clean baseline:  {clean_diff:+.4f}")

head_hook_exp = make_head_mask_hook(important_heads_expanded)
mlp_hook_exp  = make_mlp_mask_hook(important_mlps_expanded)

hooks_expanded = [
    (f"blocks.{l}.attn.hook_z",   head_hook_exp)
    for l in range(model.cfg.n_layers)
] + [
    (f"blocks.{l}.hook_mlp_out",  mlp_hook_exp)
    for l in range(model.cfg.n_layers)
]

logits_expanded = model.run_with_hooks(tokens_clean, fwd_hooks=hooks_expanded)
diff_expanded = get_logit_diff(logits_expanded)
pct_expanded = (diff_expanded - corr_diff) / (clean_diff - corr_diff) * 100

print(f"  Expanded circuit: {diff_expanded:+.4f}  ({pct_expanded:.1f}% of full prediction)")
print()
if diff_expanded > clean_diff:
    print("Note: the expanded circuit score EXCEEDS the full model baseline.")
    print("This happens because the full model contains components that COMPETE with")
    print("the correct answer. Removing them (by keeping only the 'pro-Paris' circuit)")
    print("produces a cleaner, stronger signal — the circuit is not diluted by noise.")

### 6.3 Circuit Generalization Test

A genuine circuit should generalise beyond the specific example it was discovered on.
Here we test the **France circuit** on other country-capital pairs to see whether the
same components are sufficient.

In [ ]:
print("Circuit Generalization Test")
print("-" * 50)
print("Testing the France-derived expanded circuit on other countries...\n")

test_countries = [
    ("Germany", " Berlin",  " Rome"),
    ("Spain",   " Madrid",  " Rome"),
    ("Japan",   " Tokyo",   " Rome"),
]

for country, capital, wrong in test_countries:
    t_clean = model.to_tokens(f"The capital of {country} is")
    tok_c = model.to_single_token(capital)
    tok_w = model.to_single_token(wrong)

    def get_diff_for(logits):
        return (logits[0, -1, tok_c] - logits[0, -1, tok_w]).item()

    # Full model baseline
    logits_base, _ = model.run_with_cache(t_clean)
    base = get_diff_for(logits_base)

    # France circuit applied to this country
    logits_circ = model.run_with_hooks(t_clean, fwd_hooks=hooks_expanded)
    circ = get_diff_for(logits_circ)

    pct = (circ / base) * 100 if base != 0 else float("nan")
    status = "✓ generalises" if pct > 50 else ("△ partial" if pct > 10 else "✗ fails")
    print(f"  {country:10s} ({capital.strip():6s}): full={base:+.2f}, circuit={circ:+.2f}, "
          f"recovery={pct:5.1f}%  {status}")

print()
print("Result: the France-derived circuit does NOT generalise well to other countries.")
print("This motivates the multi-country analysis below.")

## 7. Multi-Country Circuit Analysis

Why does the France circuit fail on Spain or Japan? To answer this, we run the full
circuit discovery pipeline for each country independently and compare the results.

This reveals which components are **universal** (shared across all countries) and which
are **specialised** (specific to certain regions or languages).

In [ ]:
import pandas as pd

print("Multi-Country Circuit Analysis")
print("=" * 50)

# Test cases: country → correct capital (wrong answer is always 'Rome' as baseline)
test_cases = {
    "France":  " Paris",
    "Spain":   " Madrid",
    "Germany": " Berlin",
    "Japan":   " Tokyo",
    "Russia":  " Moscow",
    "China":   " Beijing",
}

token_wrong = model.to_single_token(" Rome")  # shared wrong answer for comparison

circuit_data = []

for country, capital in test_cases.items():
    tok_correct = model.to_single_token(capital)
    tokens_c = model.to_tokens(f"The capital of {country} is")

    # ── Helper: logit(correct) - logit(Rome) ────────────────────────────────
    def get_local_diff(logits):
        return (logits[0, -1, tok_correct] - logits[0, -1, token_wrong]).item()

    # ── Baseline ────────────────────────────────────────────────────────────
    logits_base, cache_c = model.run_with_cache(tokens_c)
    base_diff = get_local_diff(logits_base)

    # ── Find mover heads (same attention scan as above) ──────────────────────
    attn_subj = torch.zeros((n_layers, n_heads))
    for l in range(n_layers):
        pattern = cache_c[f"blocks.{l}.attn.hook_pattern"][0]
        attn_subj[l] = pattern[:, -1, -2].detach().cpu()

    mover_cands = torch.nonzero(attn_subj > 0.2, as_tuple=False)

    # ── Ablate candidate heads → keep those that matter ──────────────────────
    important_heads = []
    for l, h in mover_cands:
        logits_abl = model.run_with_hooks(
            tokens_c,
            fwd_hooks=[(f"blocks.{l}.attn.hook_z", make_z_ablation_hook(h))]
        )
        if base_diff - get_local_diff(logits_abl) > 0.1:
            important_heads.append(f"L{l}H{h}")

    # ── Ablate MLP layers → keep those that matter ───────────────────────────
    important_mlps = []
    for layer in range(n_layers):
        logits_abl = model.run_with_hooks(
            tokens_c,
            fwd_hooks=[(f"blocks.{layer}.hook_mlp_out", mlp_ablation_hook)]
        )
        if base_diff - get_local_diff(logits_abl) > 0.1:
            important_mlps.append(f"MLP{layer}")

    circuit_data.append({
        "Country":        country,
        "Capital":        capital.strip(),
        "Baseline Diff":  round(base_diff, 2),
        "Important Heads": ", ".join(important_heads),
        "Important MLPs":  ", ".join(important_mlps),
    })
    print(f"  {country}: done (baseline={base_diff:+.2f})")

print()
df = pd.DataFrame(circuit_data)
display(df)

### 7.1: Why Activation Patching is Fragile — and What to Do About It

The generalisation test above reveals a deeper methodological problem with activation patching.

**The core issue:** patching is sensitive to the choice of *corrupted input*.

When we swap "France" → "Italy", we are making a very specific perturbation. The circuit we discover is only guaranteed to be responsible for the *difference* between those two particular prompts. Change the corruption (e.g. use a nonsense token, or scramble word order), and you may find a completely different set of "important" components — because you are measuring a different causal contrast.

In other words: **the discovered circuit is an artefact of the experimental setup as much as a property of the model**.

| Limitation | Description |
|---|---|
| Corpus-specific | Results depend on the choice of clean/corrupted pair |
| Binary | A component is either "in" or "out" — no notion of degree |
| Non-additive | Ablating heads independently ignores interactions between them |
| Hard to scale | Requires a separate experiment for every task and every corruption |

---

**A more robust alternative: attribution-based methods.**

Rather than intervening on specific activations, *attribution* methods assign a continuous importance score to every component by analysing the model's computation graph analytically — no perturbations required.

One particularly clean recent approach is **Information Flow Routes** (Ferrando et al., 2024, *"A primer on the inner workings of transformer-based language models"*). It decomposes the contribution of every attention head and MLP layer to the final logit directly from the residual stream, using a technique called **logit lens attribution**:

$$
\text{logit}(w) = \underbrace{x_0 W_U}_\text{embedding} + \sum_{\ell=0}^{L} \underbrace{\Delta_\ell^{\text{attn}} W_U + \Delta_\ell^{\text{MLP}} W_U}_\text{layer }\ell\text{ contributions}
$$

where $\Delta_\ell^{\text{attn}}$ and $\Delta_\ell^{\text{MLP}}$ are the additive updates each component deposits into the residual stream, and $W_U$ is the unembedding matrix.

**Advantages over patching:**

| Property | Activation Patching | Information Flow Routes |
|---|---|---|
| Requires a corrupted input? | ✓ Yes | ✗ No |
| Sensitive to corruption choice? | ✓ Highly | ✗ No |
| Assigns continuous importance scores? | ✗ No (binary ablation) | ✓ Yes |
| Captures head–head interactions? | ✗ Only partially | ✓ Via attention rollout |
| Scales to large models? | ✗ Expensive (one run per component) | ✓ Single forward pass |

Patching remains a valuable *causal* tool — it can confirm that a component is *sufficient* for a behaviour. But for **exploration and hypothesis generation**, attribution-based methods give a much more stable, generalizable picture of how information flows.

> **Further reading:** Ferrando et al. (2024), *"A Primer on the Inner Workings of Transformer-based Language Models"*, arXiv 2405.00208.


## 8. Conclusion: The Anatomy of Factual Retrieval

By running the circuit discovery pipeline across six countries, a clear picture emerges:

### 8.1 Universal Routing Infrastructure

Every country in our test set relies on the **same core components**:
- **Attention heads L2_H9, L3_H6, L9_H8** — the universal "fact delivery" highway
- **MLP Layer 1** — early subject processing, present in all cases

These components form the backbone of the capital-city retrieval circuit, regardless of
which country is being queried.

### 8.2 Distributed Factual Storage

The **specific MLP layers** that store the capital city differ by region:

| Group | Key MLP(s) |
|-------|------------|
| France, Germany | MLP 8 |
| Japan, Russia, China | MLP 5 |
| Spain | MLP 0 |

This suggests the model organises factual knowledge geographically/linguistically across
different MLP layers — each acting as a specialised "filing cabinet."

### 8.3 Why the France Circuit Failed to Generalise

Our France circuit included MLP 8 (where France→Paris is stored) but excluded MLP 5
(where Japan→Tokyo is stored). When we applied the France circuit to Japan, we locked
the model out of the one MLP it needed.

### 8.4 The Big Picture

Neural networks store knowledge not in a single, localised place, but in a **dynamic
Markov blanket structure**:

```
Input tokens
    ↓
[Early layers 0-3] — Subject identification & early knowledge retrieval (MLP 1, 2, 3)
    ↓                  at the SUBJECT token position
[Mid layers 4-8]   — Specialised fact storage (MLP 5 or MLP 8 depending on the fact)
    ↓
[Layer 9]          — ★ CAUSAL BOTTLENECK: information moves from subject → last token
    ↓                  (attention head L9_H8 is the key mover)
[Layers 10-11]     — Final refinement at the last token position
    ↓
Output: "Paris"
```

Mechanistic interpretability gives us the tools to untangle this beautiful, complex web —
and ultimately to understand, predict, and control what large language models know and do.

## 9. BONUS DIY: Arithmetic Processing in LLMs

You've now mastered the core tools of mechanistic interpretability applied to factual
retrieval. As an extension, apply the same techniques to **arithmetic processing**.

Mathematical operations provide an interesting contrast to country-capital recall:
- Factual recall: the answer is a specific token stored in MLP weights
- Arithmetic: the answer must be *computed* from numerical tokens

**Your task:** replicate the analysis above for simple addition problems.

Some suggested prompts:
```python
prompt_clean = "2 + 3 ="
prompt_corr  = "2 + 4 ="   # corrupted: different sum
token_correct = model.to_single_token(" 5")
token_wrong   = model.to_single_token(" 6")
```

**Guiding questions:**
1. Which layers show the largest causal effect in the activation patching heatmap?
   Does the pattern look similar to factual retrieval, or different?
2. Are the same attention heads (L2_H9, L3_H6, L9_H8) important for arithmetic,
   or does the model use a different circuit?
3. Do MLP layers play a bigger or smaller role compared to factual recall?
4. Try a range of sums (e.g. 1+1, 3+4, 7+8). Is the same circuit used for all,
   or does the circuit change with the magnitude of the numbers?

**Hint:** For arithmetic, you may need to be careful about tokenisation.
Check how the model tokenises multi-digit numbers:
```python
model.to_str_tokens("12 + 7 = 19")
```

In [ ]:
# ── Starter code for the arithmetic DIY exercise ────────────────────────────
# Fill in the blanks and run the same experiments as above!

# Define your arithmetic prompts
arith_clean = "2 + 3 ="
arith_corr  = "2 + 4 ="

# Check tokenisation — arithmetic tokens may surprise you!
print("Arithmetic prompt tokens:")
print(f"  Clean:     {model.to_str_tokens(arith_clean)}")
print(f"  Corrupted: {model.to_str_tokens(arith_corr)}")

# TODO: define target tokens and a logit-diff function for arithmetic
# TODO: run run_with_cache on both prompts
# TODO: perform the layer-by-layer scan
# TODO: generate the full causal heatmap
# TODO: identify important attention heads and MLP layers
# TODO: compare with the France-capital circuit

print("\nGood luck! Use the cells above as a template.")

### Interpretability Libraries

- **`nnsight`**: A Python library for inspecting and intervening in neural network computations, especially useful for transformer models. It enables tracing activations, modifying internal states, and running causal interventions directly during the forward pass.

- **`inseq`**: A library for interpreting sequence generation models (e.g., LLMs, translation models) using attribution methods such as Integrated Gradients, DeepLIFT, and attention-based analyses. It provides tools to analyze token-level contributions in generated text.

These tools make it easier to understand how models process inputs and produce outputs.